<a href="https://colab.research.google.com/github/dorian-goueytes/M1_SCE_TT_signal_S2/blob/main/artifact_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Détection d'artefact via une step-function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from matplotlib.animation import FuncAnimation
from matplotlib import rc

## Create a signal including an artifact

In [ ]:
fs = 100                      # sampling rate (Hz)
duration = 2                  # seconds
n_samples = fs * duration
time = np.arange(n_samples) / fs


np.random.seed(42)

white_noise = np.random.randn(n_samples)

# Bandpass filter 1–30 Hz to mimic EEG spectrum
def bandpass(data, low, high, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, data)

eeg_background = bandpass(white_noise, 1, 30, fs)
eeg_background *= 10  # scale to realistic amplitude (~10 µV)

artifact = np.zeros(n_samples)
artifact_start = int(0.5 * fs)
artifact_duration = int(0.3 * fs)
artifact_amplitude = 80  # eye artifacts are large (~50–200 µV)

artifact[artifact_start:artifact_start + artifact_duration] = artifact_amplitude


eeg_signal = eeg_background + artifact


plt.figure(figsize=(10,4))
plt.plot(time, eeg_signal, label='EEG + eye artifact')
plt.xlabel("Time (s)")
plt.ylabel("Amplitude (µV)")
plt.title("Simulated EEG Trial with Boxcar Eye Movement Artifact")
plt.legend()
plt.tight_layout()
plt.show()

## Create a 200ms kernel Step-function

In [ ]:
kernel_duration = 0.2
kernel_size = int(kernel_duration * fs)  # half duration (~150 ms)
step_kernel = np.concatenate([
    -np.ones(kernel_size),
    np.ones(kernel_size)
])

# Normalize kernel
step_kernel = step_kernel / np.sum(np.abs(step_kernel))

plt.figure()
plt.title("Convolution Kernel")
plt.plot(np.linspace(0,kernel_duration,len(step_kernel)), step_kernel)
plt.xlabel("Time (ms)")
plt.ylabel("Amplitude")
plt.show()

## Convolution of signal with step function

In [ ]:
conv_output = np.convolve(eeg_signal, step_kernel, mode='same')

# Threshold detection
threshold = 2.5 * np.std(conv_output)
detections = np.abs(conv_output) > threshold


plt.figure(figsize=(12,6))

plt.subplot(2,1,1)
plt.plot(time, eeg_signal)
plt.title("Simulated EEG Signal")
plt.ylabel("Amplitude (µV)")



plt.subplot(2,1,2)
plt.plot(time, conv_output, label='Convolution Output')
plt.axhline(threshold, color='r', linestyle='--')
plt.axhline(-threshold, color='r', linestyle='--')
plt.title("Step-Function Convolution (Artifact Detection)")
plt.xlabel("Time (s)")
plt.ylabel("Filter Response")
plt.legend()


plt.tight_layout()
plt.show()

## Convolution Animation

In [ ]:
rc('animation', html='jshtml')

N = fs * duration
# normalize (optional but cleaner interpretation)
step_kernel = step_kernel / np.sum(np.abs(step_kernel))

convolved_signal = np.convolve(eeg_signal, step_kernel, mode='same')


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6))
ax1.plot(time, eeg_signal, alpha=0.4, label="EEG signal")
# kernel line (actual shape)
kernel_line, = ax1.plot([], [], lw=2, label="Step kernel")

ax1.set_xlim(time[0], time[-1])
ax1.set_ylim(eeg_signal.min()*1.2,
             eeg_signal.max()*1.2)

ax1.set_title("Sliding Step Kernel")
ax1.set_ylabel("Amplitude")
ax1.grid()
ax1.legend()

# Convolution output
conv_line, = ax2.plot([], [],
                      lw=2,
                      label="Convolution output")

ax2.set_xlim(time[0], time[-1])
ax2.set_ylim(convolved_signal.min()*1.2,
             convolved_signal.max()*1.2)

ax2.set_title("Convolution Result")
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Amplitude")
ax2.grid()
ax2.legend()

def update(frame):

    start = frame - len(step_kernel)//2
    idx = np.arange(len(step_kernel)) + start
    valid = (idx >= 0) & (idx < N)
    t_kernel = time[idx[valid]]
    # scale kernel for visibility
    scaled_kernel = step_kernel[valid] * np.max(np.abs(eeg_signal)) * 3
    kernel_line.set_data(t_kernel, scaled_kernel)
    conv_line.set_data(time[:frame],convolved_signal[:frame])

    return kernel_line, conv_line


anim = FuncAnimation(fig, update, frames=N,interval=40,blit=True)
plt.close(fig)
anim